# Moderador de Chat con BERT Multilingüe - Versión Avanzada
Este notebook implementa un modelo de moderación de texto (Odio vs No Odio) utilizando BERT multilingüe, optimizado para evitar el overfitting:
- **Early Stopping**: Detiene el entrenamiento si la pérdida de validación deja de mejorar.
- **Model Checkpointing**: Restaura automáticamente el mejor estado del modelo encontrado.
- **7 Épocas** con regularización agresiva.
- **Weight Decay y Dropout** ajustados para mejor generalización.

In [ ]:
!pip install transformers scikit-learn pandas matplotlib seaborn

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertModel, BertTokenizer, get_linear_schedule_with_warmup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import copy
import time

In [ ]:
# CONFIGURACIÓN Y HIPERPARÁMETROS
RANDOM_SEED = 42
MAX_LEN = 150
BATCH_SIZE = 32
NCLASSES = 2
EPOCHS = 7           # Incrementado a 7, controlado con Early Stopping
LEARNING_RATE = 2e-5 # Learning rate estándar para BERT
WEIGHT_DECAY = 0.01  # Regularización L2
DROPOUT_PROB = 0.4   # Dropout para mitigar overfitting
PATIENCE = 2         # Épocas de espera para Early Stopping

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

In [ ]:
# CARGA DE DATOS
df = pd.read_csv("dataset_ready_nlp.csv")
df['labels'] = df['labels'].astype(int)

print(f"Total de registros: {len(df)}")
print("Distribución de clases:")
print(df['labels'].value_counts())
df.head()

In [ ]:
# TOKENIZADOR
PRE_TRAINED_MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

In [ ]:
class ModeradorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

        print("Pre-tokenizando datos...")
        self.encodings = tokenizer(
            list(texts),
            max_length=self.max_len,
            truncation=True,
            add_special_tokens=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt'
        )
        print("Tokenización completada!")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        return {
            'text': self.texts[item],
            'input_ids': self.encodings['input_ids'][item],
            'attention_mask': self.encodings['attention_mask'][item],
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

In [ ]:
# SPLIT DE DATOS
df_train, df_temp = train_test_split(
    df, test_size=0.3, random_state=RANDOM_SEED, stratify=df['labels']
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=df_temp['labels']
)

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = ModeradorDataset(
        texts=df.text.to_numpy(),
        labels=df.labels.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)

train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)
test_data_loader = create_data_loader(df_test, tokenizer, MAX_LEN, BATCH_SIZE)

In [ ]:
class BERTModerador(nn.Module):
    def __init__(self, n_classes):
        super(BERTModerador, self).__init__()
        self.bert = BertModel.from_pretrained(PRE_TRAINED_MODEL_NAME)
        self.dropout = nn.Dropout(p=DROPOUT_PROB)
        self.linear = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        return self.linear(x)

model = BERTModerador(NCLASSES)
model = model.to(device)

In [ ]:
# OPTIMIZADOR Y SCHEDULER
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss().to(device)

In [ ]:
# FUNCIONES DE ENTRENAMIENTO
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        # Mixed Precision
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())
        scheduler.step()

    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, labels)

            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = correct_predictions.double() / n_examples
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, np.mean(losses), f1, all_labels, all_preds

In [ ]:
# BUCLE DE ENTRENAMIENTO CON EARLY STOPPING Y SELECCIÓN DEL MEJOR MODELO
history = {
    'train_acc': [], 'train_loss': [],
    'val_acc': [], 'val_loss': [], 'val_f1': []
}
best_loss = float('inf')
best_model_state = None
early_stop_count = 0

for epoch in range(EPOCHS):
    print(f'\nÉpoca {epoch + 1}/{EPOCHS}')
    print('-' * 20)

    train_acc, train_loss = train_epoch(
        model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train)
    )
    val_acc, val_loss, val_f1, _, _ = eval_model(
        model, val_data_loader, loss_fn, device, len(df_val)
    )

    print(f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}')
    print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}')

    history['train_acc'].append(train_acc.item())
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc.item())
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)

    # Lógica de Early Stopping y Guardado
    if val_loss < best_loss:
        best_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        early_stop_count = 0
        print("*** Nuevo mejor modelo guardado (menor pérdida) ***")
    else:
        early_stop_count += 1
        print(f"-> La pérdida no mejoró. Early stopping count: {early_stop_count}/{PATIENCE}")

    if early_stop_count >= PATIENCE:
        print("\n!!! EARLY STOPPING activado. Deteniendo entrenamiento para evitar overfitting !!!")
        break

# Restaurar el mejor estado al finalizar
if best_model_state:
    model.load_state_dict(best_model_state)
    print("\nMejor estado del modelo restaurado para pruebas finales.")

In [ ]:
# VISUALIZACIÓN DE RESULTADOS
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss')
plt.plot(epochs_range, history['val_loss'], label='Val Loss')
plt.title('Pérdida (Loss)')
plt.xlabel('Época')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Train Acc')
plt.plot(epochs_range, history['val_acc'], label='Val Acc')
plt.title('Exactitud (Accuracy)')
plt.xlabel('Época')
plt.legend()

plt.show()

In [ ]:
# EVALUACIÓN FINAL EN TEST
test_acc, test_loss, test_f1, y_true, y_pred = eval_model(
    model, test_data_loader, loss_fn, device, len(df_test)
)

print(f"\nResultados en conjunto de TEST:")
print(f"Accuracy: {test_acc:.4f}")
print(f"F1 Score: {test_f1:.4f}")

print("\nReporte de Clasificación:")
print(classification_report(y_true, y_pred, target_names=['No Odio', 'Odio']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Odio', 'Odio'], yticklabels=['No Odio', 'Odio'])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

In [ ]:
# GUARDAR EL MODELO FINAL
torch.save(model.state_dict(), "moderador_bert_final.pt")
print("Modelo guardado como 'moderador_bert_final.pt'")

## Conclusiones
Se implementó un esquema de 7 épocas con **Early Stopping (paciencia=2)**, lo que permite al modelo buscar el punto óptimo de convergencia sin riesgo de sobreajuste excesivo. Al restaurar el mejor estado basado en la pérdida de validación, garantizamos que el modelo final sea el más capaz de generalizar ante nuevos datos.